# 01 — Preprocessing & Dataset Splits  (Local)

Runs entirely **locally** — no Google Drive, no ZIP extraction, no Colab.

| | |
|---|---|
| **Input** | `Trash_Dataset/GARBAGE CLASSIFICATION/` |
| **Split output** | `data/processed/dataset_split_70_15_15/` |
| **Crops output** | `data/processed/dataset_crops/` |
| **Figures** | `figures/preprocessing/` |
| **Reports** | `results/metrics/01_cleaning_report.csv` · `02_split_stats.csv` · `03_crop_metadata.csv` |

**Run cells in order — each builds on the previous.**

In [ ]:
# ── USER CONFIGURATION ────────────────────────────────────────────────────────
# Auto-detects repo root from notebook location (works in VS Code / Jupyter).
from pathlib import Path

_here = Path().resolve()
REPO_ROOT = _here.parent if _here.name == "notebooks" else _here

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"Exists    : {REPO_ROOT.exists()}")
# Uncomment & set manually if auto-detect is wrong:
# REPO_ROOT = Path(r"C:\Users\zaki\Desktop\aly eui\Computer_Vision_Latest")
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
# ── SETUP: IMPORTS · PATHS · LOGGING ─────────────────────────────────────────
import sys, os, hashlib, shutil, yaml, re
from collections import Counter, defaultdict

sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — saves figures without display
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.utils.seed import set_seed, SEED
from src.utils.paths import CLASSES, IDX_TO_CLASS
from src.utils.logger import get_logger
from src.data.split_dataset import build_image_inventory, make_split, get_dominant_class

set_seed()
logger = get_logger("preprocessing")

# ── Local path constants ──────────────────────────────────────────────────────
DATASET_ROOT = REPO_ROOT / "Trash_Dataset" / "GARBAGE CLASSIFICATION"
SPLIT_OUT    = REPO_ROOT / "data" / "processed" / "dataset_split_70_15_15"
CROPS_OUT    = REPO_ROOT / "data" / "processed" / "dataset_crops"
FIG_DIR      = REPO_ROOT / "figures" / "preprocessing"
METRICS_DIR  = REPO_ROOT / "results" / "metrics"

for _d in [SPLIT_OUT, CROPS_OUT, FIG_DIR, METRICS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# Class colour palette
CLASS_COLORS = {
    "BIODEGRADABLE": "#2ecc71",
    "CARDBOARD":     "#e67e22",
    "GLASS":         "#3498db",
    "METAL":         "#95a5a6",
    "PAPER":         "#f5cba7",
    "PLASTIC":       "#9b59b6",
}
HEX_LIST = [CLASS_COLORS[c] for c in CLASSES]

# ── Verify dataset present ────────────────────────────────────────────────────
assert DATASET_ROOT.exists(), f"Dataset not found: {DATASET_ROOT}"
logger.info(f"Dataset root: {DATASET_ROOT}")

print("All imports OK")
print(f"  Dataset  : {DATASET_ROOT}")
print(f"  Split out: {SPLIT_OUT}")
print(f"  Crops out: {CROPS_OUT}")
print(f"  Figures  : {FIG_DIR}")
print(f"  Metrics  : {METRICS_DIR}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 — DATASET AUDIT: structural scan · YAML forensics · class mapping
# ══════════════════════════════════════════════════════════════════════════════
root      = DATASET_ROOT
img_exts  = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
all_files = list(root.rglob("*"))

all_images = [f for f in all_files if f.is_file() and f.suffix.lower() in img_exts]
all_labels = [f for f in all_files if f.is_file() and f.suffix.lower() == ".txt"
              and "README" not in f.name]

print("=" * 70)
print("CELL 3 — DATASET AUDIT")
print("=" * 70)
print(f"  Total images : {len(all_images)}")
print(f"  Label files  : {len(all_labels)}")

print(f"\n  Vendor split (Roboflow):")
for _sp in ["train", "valid", "test"]:
    _ni = len(list((root / _sp / "images").glob("*.jpg")))
    _nl = len(list((root / _sp / "labels").glob("*.txt")))
    print(f"    {_sp:<8}: {_ni:>5} images  {_nl:>5} labels")

# ── YAML forensics ────────────────────────────────────────────────────────────
print(f"\n  YAML metadata:")
yaml_path = DATASET_ROOT / "data.yaml"
with open(yaml_path) as _f:
    yaml_data = yaml.safe_load(_f)
print(f"    nc    = {yaml_data.get('nc')}")
print(f"    names = {yaml_data.get('names')}")

# ── Filename-inferred class map ───────────────────────────────────────────────
_fname_cls = defaultdict(list)
_pattern   = re.compile(r"^([a-zA-Z]+)\d+")
for _tf in all_labels:
    _stem = _tf.stem.lower().split(".rf.")[0]
    _m    = _pattern.match(_stem)
    if not _m:
        continue
    try:
        _lines = _tf.read_text().strip().splitlines()
        if _lines:
            _cid = int(_lines[0].split()[0])
            _fname_cls[_cid].append(_m.group(1))
    except Exception:
        pass

print(f"\n  Inferred class map (filename prefix → class ID):")
for _cid in sorted(_fname_cls):
    _mc = Counter(_fname_cls[_cid]).most_common(1)[0][0]
    print(f"    Class {_cid}: '{_mc}'  ({len(_fname_cls[_cid])} samples)")

# ── GO / NO-GO ────────────────────────────────────────────────────────────────
_yaml_names = yaml_data.get("names", [])
_yaml_cls   = ({i: n.upper() for i, n in enumerate(_yaml_names)}
               if isinstance(_yaml_names, list)
               else {int(k): str(v).upper() for k, v in _yaml_names.items()})
_expected   = {i: c for i, c in enumerate(CLASSES)}
_matches    = sum(1 for i, c in _expected.items() if _yaml_cls.get(i, "").upper() == c)
print(f"\n  GO/NO-GO: {_matches}/6 classes match project spec")
if _matches == 6:
    print("  PASS — dataset classes match specification")
else:
    print("  FAIL — class mismatch; review required")

# ── Fig_01: bounding-box count per class per vendor split ─────────────────────
_scc = {}
for _sp in ["train", "valid", "test"]:
    _lbl_dir = root / _sp / "labels"
    _counts  = Counter()
    for _lf in _lbl_dir.glob("*.txt"):
        try:
            for _ln in _lf.read_text().strip().splitlines():
                _p = _ln.split()
                if _p:
                    _counts[int(_p[0])] += 1
        except Exception:
            pass
    _scc[_sp] = _counts

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
fig.suptitle("Fig_01 — Raw Dataset: Bounding-Box Count per Class per Vendor Split",
             fontsize=13, fontweight="bold")
for _ax, _sp in zip(axes, ["train", "valid", "test"]):
    _vals = [_scc[_sp].get(i, 0) for i in range(6)]
    _bars = _ax.bar(range(6), _vals, color=HEX_LIST, edgecolor="black")
    _ax.set_title(f"{_sp.upper()}  ({sum(_vals):,} boxes)", fontweight="bold")
    _ax.set_xticks(range(6))
    _ax.set_xticklabels(CLASSES, rotation=30, ha="right", fontsize=9)
    _ax.set_ylabel("Box count")
    for _b in _bars:
        _h = _b.get_height()
        if _h > 0:
            _ax.annotate(str(int(_h)),
                         xy=(_b.get_x() + _b.get_width() / 2, _h),
                         xytext=(0, 3), textcoords="offset points",
                         ha="center", va="bottom", fontsize=8)
plt.tight_layout()
_fig1 = FIG_DIR / "Fig_01_raw_class_distribution.png"
plt.savefig(_fig1, dpi=150, bbox_inches="tight")
plt.close()
print(f"\n  Saved: {_fig1.name}")
print("\n" + "=" * 70)
print("CELL 3 COMPLETE")
print("=" * 70)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 — DEEP CLEANING: zero-area boxes · image integrity · MD5 dedup
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("CELL 4 — DEEP CLEANING")
print("=" * 70)

# ── 4.1 Build master inventory ────────────────────────────────────────────────
print("\n  Building master inventory...")
df_raw = build_image_inventory(DATASET_ROOT)
print(f"  Matched image-label pairs: {len(df_raw)}")

# ── 4.2 Check for zero-area boxes ─────────────────────────────────────────────
print("\n  Checking for zero-area bounding boxes...")
zero_area_log        = []
cleaned_label_cache  = {}   # stem -> (img_path, lbl_path, [good lines])

for _, _row in df_raw.iterrows():
    _lbl  = Path(_row["label"])
    _lines = _lbl.read_text().strip().splitlines()
    _good  = []
    for _ln in _lines:
        _parts = _ln.strip().split()
        if len(_parts) == 5:
            try:
                _w, _h = float(_parts[3]), float(_parts[4])
                if _w > 0 and _h > 0:
                    _good.append(_ln)
                else:
                    zero_area_log.append({"image": _row["stem"], "line": _ln})
            except ValueError:
                pass
    cleaned_label_cache[_row["stem"]] = (_row["image"], _lbl, _good)

print(f"  Zero-area boxes found: {len(zero_area_log)}")

# ── 4.3 Image integrity check ─────────────────────────────────────────────────
print("\n  Checking image integrity...")
corrupt_stems = set()
for _stem, (_img, _lbl, _) in cleaned_label_cache.items():
    try:
        Image.open(_img).verify()
    except Exception:
        corrupt_stems.add(_stem)
print(f"  Corrupt / unreadable: {len(corrupt_stems)}")

# ── 4.4 MD5 deduplication ────────────────────────────────────────────────────
print("\n  Running MD5 deduplication...")
_md5_map = defaultdict(list)
for _stem, (_img, _, _) in cleaned_label_cache.items():
    if _stem in corrupt_stems:
        continue
    try:
        _h = hashlib.md5(Path(_img).read_bytes()).hexdigest()
        _md5_map[_h].append(_stem)
    except Exception:
        corrupt_stems.add(_stem)

dup_stems = {_s for _stems in _md5_map.values() for _s in _stems[1:]}
print(f"  Exact duplicates: {len(dup_stems)}")

# ── 4.5 Build clean pool ──────────────────────────────────────────────────────
_skip = corrupt_stems | dup_stems
_clean_records = []
for _stem in sorted(cleaned_label_cache.keys()):
    if _stem in _skip:
        continue
    _img, _lbl, _good = cleaned_label_cache[_stem]
    if not _good:          # all boxes were zero-area
        continue
    _dom = get_dominant_class(Path(_lbl))
    if _dom == -1:
        continue
    _clean_records.append({"stem": _stem, "image": _img, "label": _lbl,
                            "dominant_class": _dom})

df_clean  = pd.DataFrame(_clean_records)
n_removed = len(df_raw) - len(df_clean)

print(f"\n  Raw pool      : {len(df_raw)}")
print(f"  Corrupt       : {len(corrupt_stems)}")
print(f"  Duplicates    : {len(dup_stems)}")
print(f"  Zero-area only: {len(zero_area_log)}")
print(f"  Clean pool    : {len(df_clean)}")
print(f"  Removed total : {n_removed}")

# ── 4.6 Cleaning report CSV ───────────────────────────────────────────────────
pd.DataFrame([
    {"stage": "zero_area_boxes_removed", "count": len(zero_area_log)},
    {"stage": "corrupt_images",          "count": len(corrupt_stems)},
    {"stage": "duplicate_images",        "count": len(dup_stems)},
    {"stage": "total_removed",           "count": n_removed},
    {"stage": "clean_pool_size",         "count": len(df_clean)},
]).to_csv(METRICS_DIR / "01_cleaning_report.csv", index=False)
print(f"\n  Saved: 01_cleaning_report.csv")

# ── 4.7 Fig_02: before/after dominant-class distribution ──────────────────────
_raw_dist   = Counter(df_raw["dominant_class"].tolist())
_clean_dist = Counter(df_clean["dominant_class"].tolist())
_x = np.arange(6)
_w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
_b1 = ax.bar(_x - _w/2, [_raw_dist.get(i, 0)   for i in range(6)], _w,
             label="Before cleaning", color=HEX_LIST, edgecolor="black", alpha=0.55)
_b2 = ax.bar(_x + _w/2, [_clean_dist.get(i, 0) for i in range(6)], _w,
             label="After cleaning",  color=HEX_LIST, edgecolor="black")
ax.set_xlabel("Class", fontweight="bold")
ax.set_ylabel("Image count (dominant class)", fontweight="bold")
ax.set_title("Fig_02 — Before vs After Cleaning: Dominant-Class Distribution",
             fontsize=13, fontweight="bold")
ax.set_xticks(_x)
ax.set_xticklabels(CLASSES, rotation=20, ha="right")
ax.legend()
for _b in list(_b1) + list(_b2):
    _h = _b.get_height()
    if _h > 0:
        ax.annotate(str(int(_h)),
                    xy=(_b.get_x() + _b.get_width() / 2, _h),
                    xytext=(0, 2), textcoords="offset points",
                    ha="center", va="bottom", fontsize=7)
plt.tight_layout()
_fig2 = FIG_DIR / "Fig_02_before_after_cleaning.png"
plt.savefig(_fig2, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {_fig2.name}")
print("\n" + "=" * 70)
print("CELL 4 COMPLETE")
print("=" * 70)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 — STRATIFIED 70/15/15 SPLIT + data.yaml
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("CELL 5 — STRATIFIED 70/15/15 SPLIT")
print("=" * 70)

DATA_YAML = SPLIT_OUT / "data.yaml"

if DATA_YAML.exists():
    print(f"  Split already exists — skipping file copy.")
    print(f"  Delete {SPLIT_OUT} and re-run to regenerate.")
else:
    make_split(df_clean, SPLIT_OUT)
    data_yaml_cfg = {
        "path":  str(SPLIT_OUT),
        "train": "train/images",
        "val":   "val/images",
        "test":  "test/images",
        "nc":    6,
        "names": CLASSES,
    }
    with open(DATA_YAML, "w") as _f:
        yaml.dump(data_yaml_cfg, _f, default_flow_style=None, sort_keys=False)
    print(f"  data.yaml written -> {DATA_YAML}")

# ── Print split sizes ──────────────────────────────────────────────────────────
print(f"\n  {'Split':<8} {'Images':>8} {'Labels':>8}")
print("  " + "-" * 28)
_total_imgs = 0
for _sp in ["train", "val", "test"]:
    _ni = len(list((SPLIT_OUT / _sp / "images").glob("*.jpg")))
    _nl = len(list((SPLIT_OUT / _sp / "labels").glob("*.txt")))
    _total_imgs += _ni
    _flag = "" if _ni == _nl else "  <- MISMATCH"
    print(f"  {_sp:<8} {_ni:>8} {_nl:>8}{_flag}")
print(f"  {'TOTAL':<8} {_total_imgs:>8}")

# ── Per-class dominant distribution + CSV ─────────────────────────────────────
print(f"\n  Per-class image count (dominant class) per split:")
_split_rows = []
for _sp in ["train", "val", "test"]:
    _lbl_dir    = SPLIT_OUT / _sp / "labels"
    _dom_counts = Counter()
    for _lf in _lbl_dir.glob("*.txt"):
        try:
            _d = get_dominant_class(_lf)
            if _d != -1:
                _dom_counts[_d] += 1
        except Exception:
            pass
    _tot = sum(_dom_counts.values()) or 1
    print(f"\n  {_sp.upper()}:")
    for _i, _cls in enumerate(CLASSES):
        _cnt = _dom_counts.get(_i, 0)
        print(f"    {_cls:<16} {_cnt:>5}  ({100*_cnt/_tot:4.1f}%)")
        _split_rows.append({"split": _sp, "class": _cls, "dominant_images": _cnt})

_split_df = pd.DataFrame(_split_rows)
_split_df.to_csv(METRICS_DIR / "02_split_stats.csv", index=False)
print(f"\n  Saved: 02_split_stats.csv")

# ── Fig_03: grouped + stacked split distribution ──────────────────────────────
_pivot = (_split_df.pivot(index="split", columns="class", values="dominant_images")
          .reindex(["train", "val", "test"])[CLASSES])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Fig_03 — Stratified 70/15/15 Split: Class Distribution",
             fontsize=13, fontweight="bold")

_x = np.arange(6)
_w = 0.25
for _i, (_sp, _col) in enumerate(zip(["train", "val", "test"],
                                     ["#2980b9", "#27ae60", "#e74c3c"])):
    _vals = [_pivot.loc[_sp, _cls] for _cls in CLASSES]
    axes[0].bar(_x + _i*_w, _vals, _w, label=_sp.capitalize(),
                color=_col, edgecolor="black")
axes[0].set_xticks(_x + _w)
axes[0].set_xticklabels(CLASSES, rotation=20, ha="right")
axes[0].set_ylabel("Image count")
axes[0].set_title("Grouped — per class per split")
axes[0].legend()

_pivot.plot(kind="bar", stacked=True, ax=axes[1], color=HEX_LIST, edgecolor="black")
axes[1].set_xticklabels(["Train", "Val", "Test"], rotation=0)
axes[1].set_ylabel("Image count")
axes[1].set_title("Stacked — total images per split")
axes[1].legend(title="Class", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
_fig3 = FIG_DIR / "Fig_03_split_class_distribution.png"
plt.savefig(_fig3, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {_fig3.name}")
print("\n" + "=" * 70)
print("CELL 5 COMPLETE")
print("=" * 70)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 — VISUAL VERIFICATION: bbox overlay · vendor-split corruption check
# ══════════════════════════════════════════════════════════════════════════════
import random as _random
_random.seed(SEED)

print("=" * 70)
print("CELL 6 — VISUAL VERIFICATION")
print("=" * 70)

# ── 6.1 Sample train images with ground-truth boxes ───────────────────────────
_train_imgs = list((SPLIT_OUT / "train" / "images").glob("*.jpg"))
_train_lbls = SPLIT_OUT / "train" / "labels"
_samples    = _random.sample(_train_imgs, min(5, len(_train_imgs)))

fig, axes = plt.subplots(1, 5, figsize=(25, 6))
fig.suptitle("Fig_04 — Random Train Samples with Ground-Truth Bounding Boxes",
             fontsize=13, fontweight="bold")

for _ai, _img_path in enumerate(_samples):
    _stem    = _img_path.stem
    _lbl_path = _train_lbls / f"{_stem}.txt"
    _img     = Image.open(_img_path).convert("RGB")
    _draw    = ImageDraw.Draw(_img)
    _iw, _ih = _img.size
    _boxes   = []

    if _lbl_path.exists():
        for _ln in _lbl_path.read_text().strip().splitlines():
            _parts = _ln.split()
            if len(_parts) == 5:
                try:
                    _cls = int(_parts[0])
                    _xc, _yc, _bw, _bh = map(float, _parts[1:])
                    _x1 = max(0, int((_xc - _bw/2) * _iw))
                    _y1 = max(0, int((_yc - _bh/2) * _ih))
                    _x2 = min(_iw, int((_xc + _bw/2) * _iw))
                    _y2 = min(_ih, int((_yc + _bh/2) * _ih))
                    _color = HEX_LIST[_cls % 6]
                    _draw.rectangle([_x1, _y1, _x2, _y2], outline=_color, width=3)
                    _draw.text((_x1+2, _y1+2), CLASSES[_cls][:4], fill=_color)
                    _boxes.append(_cls)
                except Exception:
                    pass

    _ax = axes[_ai]
    _ax.imshow(_img)
    _summary = ", ".join(f"{CLASSES[c][:5]}" for c in set(_boxes))
    _ax.set_title(f"{_stem[:22]}
{len(_boxes)} box(es)
{_summary}", fontsize=8)
    _ax.axis("off")

plt.tight_layout()
_fig4 = FIG_DIR / "Fig_04_bbox_verification_samples.png"
plt.savefig(_fig4, dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: {_fig4.name}")

# ── 6.2 Class coverage: vendor split vs our split ─────────────────────────────
print(f"\n  Class coverage — original Roboflow split:")
for _sp in ["train", "valid", "test"]:
    _cls_ids = set()
    for _lf in (DATASET_ROOT / _sp / "labels").glob("*.txt"):
        for _ln in _lf.read_text().strip().splitlines():
            _p = _ln.split()
            if _p:
                try: _cls_ids.add(int(_p[0]))
                except ValueError: pass
    _missing = set(range(6)) - _cls_ids
    _flag    = "OK" if not _missing else f"MISSING {sorted(_missing)}"
    print(f"    {_sp:<8}: {len(_cls_ids)}/6 classes  [{_flag}]")

print(f"\n  Class coverage — our 70/15/15 split:")
for _sp in ["train", "val", "test"]:
    _cls_ids = set()
    for _lf in (SPLIT_OUT / _sp / "labels").glob("*.txt"):
        for _ln in _lf.read_text().strip().splitlines():
            _p = _ln.split()
            if _p:
                try: _cls_ids.add(int(_p[0]))
                except ValueError: pass
    _missing = set(range(6)) - _cls_ids
    _flag    = "OK" if not _missing else f"MISSING {sorted(_missing)}"
    print(f"    {_sp:<8}: {len(_cls_ids)}/6 classes  [{_flag}]")

# ── 6.3 Image / label count per split ────────────────────────────────────────
print(f"\n  Image-label pair count per split:")
print(f"  {'Split':<8} {'Images':>8} {'Labels':>8} {'Match?':>8}")
print("  " + "-" * 36)
for _sp in ["train", "val", "test"]:
    _ni = len(list((SPLIT_OUT / _sp / "images").glob("*.jpg")))
    _nl = len(list((SPLIT_OUT / _sp / "labels").glob("*.txt")))
    print(f"  {_sp:<8} {_ni:>8} {_nl:>8} {'OK' if _ni == _nl else 'MISMATCH':>8}")

print("\n" + "=" * 70)
print("CELL 6 COMPLETE")
print("=" * 70)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 — CROP EXTRACTION: 224x224 crops from ground-truth bounding boxes
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("CELL 7 — CROP EXTRACTION  (224x224)")
print("=" * 70)

CROP_SIZE       = (224, 224)
MIN_CROP_PIXELS = 16

# ── Skip if crops already exist ───────────────────────────────────────────────
_existing = list(CROPS_OUT.rglob("*.jpg"))
if _existing:
    print(f"  Crops already exist ({len(_existing):,} files) — loading metadata only.")
    print(f"  Delete {CROPS_OUT} and re-run to regenerate.")
    _crop_records = []
    for _sp in ["train", "val", "test"]:
        for _cls in CLASSES:
            _cd = CROPS_OUT / _sp / _cls
            if not _cd.exists():
                continue
            for _p in _cd.glob("*.jpg"):
                _crop_records.append({"split": _sp, "class_name": _cls,
                                      "crop_filename": _p.name,
                                      "class_id": CLASSES.index(_cls),
                                      "source_image": _p.stem.rsplit("_obj", 1)[0],
                                      "bbox_px": []})
    crop_df    = pd.DataFrame(_crop_records)
    total_crops = len(crop_df)
    rejected    = {"too_small": 0, "out_of_bounds": 0, "read_error": 0}
else:
    _crop_records = []
    total_crops   = 0
    rejected      = {"too_small": 0, "out_of_bounds": 0, "read_error": 0}

    for _sp in ["train", "val", "test"]:
        _img_dir    = SPLIT_OUT / _sp / "images"
        _lbl_dir    = SPLIT_OUT / _sp / "labels"
        _sp_count   = 0

        for _cls in CLASSES:
            (CROPS_OUT / _sp / _cls).mkdir(parents=True, exist_ok=True)

        for _img_path in sorted(_img_dir.glob("*.jpg")):
            _stem     = _img_path.stem
            _lbl_path = _lbl_dir / f"{_stem}.txt"
            if not _lbl_path.exists():
                continue

            try:
                _img        = Image.open(_img_path).convert("RGB")
                _iw, _ih    = _img.size
            except Exception:
                rejected["read_error"] += 1
                continue

            _obj_idx = 0
            for _ln in _lbl_path.read_text().strip().splitlines():
                _parts = _ln.strip().split()
                if len(_parts) != 5:
                    continue
                try:
                    _cid            = int(_parts[0])
                    _xc, _yc, _bw, _bh = map(float, _parts[1:])
                except ValueError:
                    continue

                if _cid >= len(CLASSES):
                    continue

                _x1 = max(0, int((_xc - _bw/2) * _iw))
                _y1 = max(0, int((_yc - _bh/2) * _ih))
                _x2 = min(_iw, int((_xc + _bw/2) * _iw))
                _y2 = min(_ih, int((_yc + _bh/2) * _ih))

                if (_x2 - _x1) < MIN_CROP_PIXELS or (_y2 - _y1) < MIN_CROP_PIXELS:
                    rejected["too_small"] += 1
                    continue
                if _x2 <= _x1 or _y2 <= _y1:
                    rejected["out_of_bounds"] += 1
                    continue

                _cls_name = CLASSES[_cid]
                _crop     = _img.crop((_x1, _y1, _x2, _y2)).resize(CROP_SIZE,
                                       Image.Resampling.BILINEAR)
                _fname    = f"{_stem}_obj{_obj_idx:02d}_cls{_cid}.jpg"
                _crop.save(CROPS_OUT / _sp / _cls_name / _fname, quality=95)

                _crop_records.append({
                    "split":        _sp,
                    "source_image": _stem,
                    "crop_filename": _fname,
                    "class_id":     _cid,
                    "class_name":   _cls_name,
                    "bbox_px":      [_x1, _y1, _x2, _y2],
                })
                _obj_idx    += 1
                _sp_count   += 1
                total_crops += 1

        print(f"  {_sp:<6}: {_sp_count:>6} crops extracted")

    crop_df = pd.DataFrame(_crop_records)

print(f"\n  Total crops  : {total_crops:,}")
print(f"  Too small    : {rejected['too_small']:,}  (< {MIN_CROP_PIXELS}px side)")
print(f"  Out of bounds: {rejected['out_of_bounds']:,}")
print(f"  Read errors  : {rejected['read_error']:,}")

# ── Metadata CSV ──────────────────────────────────────────────────────────────
crop_df.to_csv(METRICS_DIR / "03_crop_metadata.csv", index=False)
print(f"\n  Saved: 03_crop_metadata.csv  ({len(crop_df):,} rows)")

# ── Fig_05: crop count per class ──────────────────────────────────────────────
_overall = crop_df["class_name"].value_counts().reindex(CLASSES, fill_value=0)
fig, ax = plt.subplots(figsize=(10, 5))
_bars = ax.bar(range(6), _overall.values, color=HEX_LIST, edgecolor="black")
ax.set_title("Fig_05 — Crop Count per Class (all splits)",
             fontsize=13, fontweight="bold")
ax.set_xticks(range(6))
ax.set_xticklabels(CLASSES, rotation=20, ha="right")
ax.set_ylabel("Crop count")
for _b in _bars:
    _h = _b.get_height()
    if _h > 0:
        ax.annotate(f"{int(_h):,}",
                    xy=(_b.get_x() + _b.get_width()/2, _h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=9, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig_05_crop_count_per_class.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: Fig_05_crop_count_per_class.png")

# ── Fig_06: per-split stacked bar ──────────────────────────────────────────────
_scc = (crop_df.groupby(["split", "class_name"]).size()
        .unstack(fill_value=0)
        .reindex(["train", "val", "test"])
        .reindex(columns=CLASSES, fill_value=0))
fig, ax = plt.subplots(figsize=(8, 5))
_scc.plot(kind="bar", stacked=True, ax=ax, color=HEX_LIST, edgecolor="black")
ax.set_title("Fig_06 — Crop Count per Split (stacked by class)",
             fontsize=13, fontweight="bold")
ax.set_xticklabels(["Train", "Val", "Test"], rotation=0)
ax.set_ylabel("Crop count")
ax.legend(title="Class", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig_06_crop_count_per_split.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"  Saved: Fig_06_crop_count_per_split.png")

# ── Fig_07: sample crops grid (6 classes x 6 samples) ────────────────────────
fig, axes = plt.subplots(6, 6, figsize=(18, 18))
fig.suptitle("Fig_07 — Sample Crops: 6 per Class (train split)",
             fontsize=14, fontweight="bold")

for _ri, _cls in enumerate(CLASSES):
    _crop_dir = CROPS_OUT / "train" / _cls
    _all_crops = sorted(_crop_dir.glob("*.jpg")) if _crop_dir.exists() else []
    _samples   = _all_crops[:6]

    for _ci, _cp in enumerate(_samples):
        _ax = axes[_ri, _ci]
        try:
            _ax.imshow(Image.open(_cp))
        except Exception:
            _ax.text(0.5, 0.5, "err", ha="center", va="center", transform=_ax.transAxes)
        _ax.set_title(f"{_cls}\n{_cp.stem[-18:]}", fontsize=6)
        _ax.axis("off")

    for _ci in range(len(_samples), 6):
        axes[_ri, _ci].axis("off")

plt.tight_layout()
plt.savefig(FIG_DIR / "Fig_07_sample_crops_per_class.png", dpi=120, bbox_inches="tight")
plt.close()
print(f"  Saved: Fig_07_sample_crops_per_class.png")

print("\n" + "=" * 70)
print("CELL 7 COMPLETE")
print("=" * 70)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 — FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
_train_n = len(list((SPLIT_OUT / "train" / "images").glob("*.jpg")))
_val_n   = len(list((SPLIT_OUT / "val"   / "images").glob("*.jpg")))
_test_n  = len(list((SPLIT_OUT / "test"  / "images").glob("*.jpg")))

print("=" * 70)
print("PREPROCESSING COMPLETE")
print("=" * 70)
print(f"  Dataset pool:")
print(f"    Raw inventory  : {len(df_raw)}")
print(f"    After cleaning : {len(df_clean)}  (removed {len(df_raw)-len(df_clean)})")
print()
print(f"  Split (70 / 15 / 15):")
print(f"    train  : {_train_n}")
print(f"    val    : {_val_n}")
print(f"    test   : {_test_n}")
print()
print(f"  Crops (224x224):")
print(f"    extracted : {total_crops:,}")
print(f"    rejected  : {sum(rejected.values()):,}")
print()
print("  Files written:")
print("    data/processed/dataset_split_70_15_15/  (YOLO split + data.yaml)")
print("    data/processed/dataset_crops/           (per-class crop folders)")
print("    results/metrics/01_cleaning_report.csv")
print("    results/metrics/02_split_stats.csv")
print("    results/metrics/03_crop_metadata.csv")
print()
print("  Figures in figures/preprocessing/:")
print("    Fig_01  raw class distribution (vendor split)")
print("    Fig_02  before / after cleaning")
print("    Fig_03  70/15/15 split class distribution")
print("    Fig_04  bbox verification samples")
print("    Fig_05  crop count per class")
print("    Fig_06  crop count per split")
print("    Fig_07  sample crops grid")
print()
print("  Next steps:")
print("    notebooks/02_yolo_baseline_E1.ipynb         -- uses data.yaml from this step")
print("    notebooks/03_classical_features_E2.ipynb    -- uses crops from this step")